# Analisi delle Metriche di Classificazione (LM Studio)

Questo notebook analizza le performance di classificazione prodotte dal modello testato tramite LM Studio. L'analisi include le principali metriche di valutazione, la matrice di confusione e una sezione dedicata all'analisi qualitativa degli errori.

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Configurazione di base per i grafici
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 6)

## Caricamento dei Dati

Carichiamo il dataset contenente le etichette originali (`original_label`) e quelle predette (`predicted_label`).

In [ ]:
data_path = '../data/labelled/lm_studio_classification.csv'
df = pd.read_csv(data_path)

print(f"Dimensioni del dataset: {df.shape}")
display(df.head())

## Metriche di Valutazione

Calcoliamo Accuracy, Precision, Recall e F1-score.

In [ ]:
y_true = df['original_label']
y_pred = df['predicted_label']

accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy globale: {accuracy:.4f}\n")

print("Report di Classificazione:")
print(classification_report(y_true, y_pred))

## Matrice di Confusione

Visualizziamo la matrice di confusione per capire meglio la distribuzione degli errori tra le classi.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=['ai', 'non_ai'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['ai', 'non_ai'], yticklabels=['ai', 'non_ai'])
plt.title('Matrice di Confusione')
plt.xlabel('Predizione')
plt.ylabel('Verità')
plt.show()

## Analisi degli Errori

Estraiamo i record in cui il modello ha fallito la classificazione per un'analisi qualitativa.

In [ ]:
# Estrazione degli errori
errori = df[df['original_label'] != df['predicted_label']]
print(f"Totale errori: {len(errori)} su {len(df)} record ({len(errori)/len(df)*100:.2f}%)\n")

# Falsi Positivi (Predetto 'ai', ma in realtà 'non_ai')
falsi_positivi = errori[(errori['predicted_label'] == 'ai') & (errori['original_label'] == 'non_ai')]
print(f"Falsi Positivi (Predetto AI, Reale NON_AI): {len(falsi_positivi)}")

# Falsi Negativi (Predetto 'non_ai', ma in realtà 'ai')
falsi_negativi = errori[(errori['predicted_label'] == 'non_ai') & (errori['original_label'] == 'ai')]
print(f"Falsi Negativi (Predetto NON_AI, Reale AI): {len(falsi_negativi)}")

### Dettaglio Falsi Positivi

Visualizziamo i record etichettati come `non_ai` ma predetti come `ai`.

In [ ]:
for index, row in falsi_positivi.iterrows():
    print(f"File: {row['filename']}")
    print(f"Testo:\n{row['text']}\n")
    print("-" * 80)

### Dettaglio Falsi Negativi

Visualizziamo i record etichettati come `ai` ma predetti come `non_ai`.

In [ ]:
for index, row in falsi_negativi.iterrows():
    print(f"File: {row['filename']}")
    print(f"Testo:\n{row['text']}\n")
    print("-" * 80)

### Conclusioni sull'Analisi Qualitativa degli Errori

L'analisi ha rivelato che **tutti i 25 errori commessi dal modello sono Falsi Negativi**. Non vi è stato alcun Falso Positivo. Questo indica che il modello è estremamente **conservativo** (Precision=1.0 per la classe AI), etichettando un testo come 'ai' solo quando ne è assolutamente certo.

Dall'ispezione dei testi misclassificati emergono 4 pattern principali che ingannano il modello:

1. **Focus su Dati e non sugli Algoritmi:** Testi che menzionano *'Big Data', 'Analytics', 'Digital Marketing'* omettendo termini espliciti come Machine Learning o Reti Neurali.
2. **Automazione Industriale e Robotica:** Progetti che descrivono hardware, sensori, IoT o *'isole robotizzate'* concentrandosi sull'ingegneria del sistema senza chiarire la componente algoritmica sottostante.
3. **Industria 4.0 generica:** Richieste di finanziamento per l'acquisto di macchinari avanzati o ampliamento di stabilimenti in ottica 4.0, dove l'AI è solo parzialmente implicita.
4. **Corsi di Formazione:** Testi relativi a percorsi formativi in cui l'Intelligenza Artificiale è solo una delle materie trattate (assieme a Cloud, Cybersecurity, ecc.), portando il modello a classificare il documento in base al suo oggetto principale (la formazione) e non alla tematica AI.